# Phase 2 — Learned detector (Colab, GPU runtime)

Trains the frozen-DINOv2 + attention-pooling detector and wires it into the fusion
pipeline. All logic lives in repo modules (`training/`, `ai_image_id/detector/`);
this notebook only orchestrates. **Runtime → Change runtime type → GPU (T4).**

Storage policy: immutable sources (GenImage zips) + small versioned artifacts
(checkpoints, calibration) in Drive; large derived caches (extracted images,
prepared splits, embedding shards) on session disk — rebuildable from the zips.
Absolute paths everywhere (`/content/...`) — cells must not depend on the cwd.

In [1]:
# ── Setup — fresh-kernel-proof ──────────────────────────────────
!apt-get -qq install -y libimage-exiftool-perl p7zip-full > /dev/null
!git clone -q https://github.com/Waranika/AI-image-Checkers.git 2>/dev/null || echo "already cloned"
%cd /content/AI-image-Checkers
%pip install -q -e .
import sys; sys.path.insert(0, "/content/AI-image-Checkers")   # exposes training/

# Verify GPU torch (Colab ships it preinstalled+matched — NEVER pip install torch here)
import torch
print(torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "→ Runtime > Change runtime type > GPU")

/content/AI-image-Checkers
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 107.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 28.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.2/51.2 kB 5.6 MB/s eta 0:00:00
  Building editable for ai-image-id (pyproject.toml) ... done
2.11.0+cu128 | cuda: True | Tesla T4


## (Optional) Smoke test — full torch chain on synthetic data, ~2 min

No downloads, no Drive. Run this on any fresh runtime before burning GPU-hours:
it exercises `prepare_split → precompute → train → calibrate` end-to-end on 60
synthetic images/class. Numbers are meaningless; execution is everything.

In [2]:
import numpy as np
from pathlib import Path
from PIL import Image

rng = np.random.default_rng(0)
for cls in ["real", "fake"]:
    d = Path(f"/content/smoke/src_{cls}"); d.mkdir(parents=True, exist_ok=True)
    for i in range(60):
        y, x = np.mgrid[0:384, 0:384]
        f1, f2 = rng.uniform(40, 140, 2)
        img = np.stack([120+60*np.sin(x/f1), 100+50*np.cos(y/f2), 90+40*np.sin((x+y)/97)], axis=-1)
        noise = rng.normal(0, 6 if cls == "real" else 2, img.shape)  # learnable class difference
        Image.fromarray(np.clip(img+noise, 0, 255).astype(np.uint8)).save(d/f"{i}.png")

from training.prepare_data import prepare_split
from training.embed import precompute
from training.train_head import train
from training.calibrate_eval import calibrate

prepare_split("/content/smoke/src_real", "/content/smoke/src_fake", "/content/smoke/train", n_per_class=48, seed=0)
prepare_split("/content/smoke/src_real", "/content/smoke/src_fake", "/content/smoke/val",   n_per_class=12, seed=1)
precompute("/content/smoke/train/manifest.csv", "/content/smoke/train", "/content/smoke/emb_train", n_aug=1, device="cuda")
precompute("/content/smoke/val/manifest.csv",   "/content/smoke/val",   "/content/smoke/emb_val",   n_aug=0, device="cuda")
ckpt = train("/content/smoke/emb_train", "/content/smoke/emb_val", out="/content/smoke/head.pt", epochs=3)
print(calibrate("/content/smoke/head.val_logits.npz", "/content/smoke/head.calibration.json"))

Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


Downloading: "https://dl.fbaipublicfiles.com/dinov2/dinov2_vits14/dinov2_vits14_pretrain.pth" to /root/.cache/torch/hub/checkpoints/dinov2_vits14_pretrain.pth


100%|██████████| 84.2M/84.2M [00:00<00:00, 334MB/s]


wrote 6 shards to /content/smoke/emb_train


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


wrote 1 shards to /content/smoke/emb_val
train: 192 samples (0.0 GB fp16), val: 24 samples, dim=384, device=cpu
epoch 0: train_loss=0.7087 val_acc=0.7500
epoch 1: train_loss=0.6197 val_acc=0.8750
epoch 2: train_loss=0.5473 val_acc=0.8750
{'temperature': 0.118, 'ece_before': 0.2648, 'ece_after': 0.1103, 'nll_before': 0.5108, 'nll_after': 0.2674}


## Step 1 — Data: GenImage SDv1.4, val-split slice

The full subset is a **30-volume split zip (~90 GB)** — exceeds session disk.
First-run strategy: extract only `val/*` (6K/class, ~3.6 GB) directly off the
Drive mount with 7z include patterns; the full `train/` run needs a
materialization route (HF mirror / bigger disk) and comes later.

One-time prerequisite in the Drive web UI: open the official GenImage folder
(link in https://github.com/GenImage-Dataset/GenImage), right-click
`stable_diffusion_v_1_4` → **Organiser → Ajouter un raccourci → Mon Drive**.
Shortcuts cost zero quota; shared-with-me files are NOT visible in Colab without one.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

SHARED = "/content/drive/MyDrive/stable_diffusion_v_1_4"

# Clean slate + sanity checks (fail here, not 30 minutes from now)
!rm -rf /content/zips /content/genimage /content/data
!df -h /content | tail -1                      # want ~60 GB available
!ls "$SHARED"/imagenet_ai_0419_sdv4.* | wc -l  # expect 30 (z01..z29 + .zip)

# Peek inside (reads the central directory only; entry rows may take a minute over FUSE)
!7z l "$SHARED/imagenet_ai_0419_sdv4.zip" | grep -E "Multivolume|Volumes|Total Physical"

# Extract ONLY val, straight off the mount — no copy. 7z chains into whichever
# volumes hold val bytes. ~3.6 GB out; minutes if val sits in the tail volumes.
!7z x -y -o/content/genimage "$SHARED/imagenet_ai_0419_sdv4.zip" \
      "imagenet_ai_0419_sdv4/val/*" | tail -4

# Verify layout BEFORE preparing — trust this output over any README diagram
!find /content/genimage -maxdepth 4 -type d
from pathlib import Path
root = Path("/content/genimage/imagenet_ai_0419_sdv4")   # adjust if find disagrees

Mounted at /content/drive
overlay         113G   47G   66G  42% /
30
Total Physical Size = 96413397770
Multivolume = +
Volumes = 30
Volumes: 30
Folders: 2
Files: 12000
Size:       3557533313
Compressed: 96413397770
/content/genimage
/content/genimage/imagenet_ai_0419_sdv4
/content/genimage/imagenet_ai_0419_sdv4/val
/content/genimage/imagenet_ai_0419_sdv4/val/ai
/content/genimage/imagenet_ai_0419_sdv4/val/nature


## Step 1b — De-confounded, disjoint splits

Single prep of the whole val pool, then **disjoint slices** of the shuffled
manifest — overlap is structurally impossible (two independent seeded draws
overlapped by ~1.7K files when we tried; measured, then fixed). The leakage
guard stays as a permanent tripwire.

In [4]:
from training.prepare_data import prepare_split
import csv, shutil
from pathlib import Path
from collections import defaultdict

# One prep of the full val pool (6000/class), single seed → files come out shuffled
prepare_split(root/"val/nature", root/"val/ai", "/content/data/all",
              n_per_class=6_000, seed=0)

# Disjoint split: first 5000/class → train, last 1000/class → val
src = Path("/content/data/all")
rows = list(csv.DictReader((src/"manifest.csv").open()))
by_label = defaultdict(list)
for r in rows:
    by_label[r["label"]].append(r)

for split, sl in (("train", slice(0, 5000)), ("val", slice(5000, 6000))):
    out = Path(f"/content/data/{split}")
    out_rows = []
    for label, rs in by_label.items():
        (out/label).mkdir(parents=True, exist_ok=True)
        for r in rs[sl]:
            shutil.move(str(src/label/r["file"]), out/label/r["file"])
            out_rows.append(r)
    with (out/"manifest.csv").open("w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=["file","label","src","jpeg_q","short_side"])
        w.writeheader(); w.writerows(out_rows)
shutil.rmtree(src)

# Leakage guard — must print 0
train_manifest = "/content/data/train/manifest.csv"
val_manifest   = "/content/data/val/manifest.csv"
train_srcs = {r["src"] for r in csv.DictReader(open(train_manifest))}
val_srcs   = {r["src"] for r in csv.DictReader(open(val_manifest))}
overlap = len(train_srcs & val_srcs)
print(f"train/val source overlap: {overlap} files", "⚠️ LEAKAGE" if overlap else "✓ clean")

train/val source overlap: 0 files ✓ clean


## Step 2 — Precompute frozen embeddings (GPU, one pass)

Val first (2K images, n_aug=0, minutes) — it doubles as the real-data smoke test.
Then train (10K × 3 variants = 30K forward passes — the long step). Shards land
on session disk (~6 GB at this scale); they're a rebuildable cache.

In [5]:
from training.embed import precompute

precompute("/content/data/val/manifest.csv",   "/content/data/val",   "/content/emb/val",
           n_aug=0, device="cuda")
precompute("/content/data/train/manifest.csv", "/content/data/train", "/content/emb/train",
           n_aug=2, device="cuda")

Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


wrote 63 shards to /content/emb/val


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


wrote 938 shards to /content/emb/train


PosixPath('/content/emb/train')

## Step 3 — Train the head (minutes) + calibrate

In [6]:
from training.train_head import train
from training.calibrate_eval import calibrate

ckpt = train("/content/emb/train", "/content/emb/val", out="/content/head.pt",
             epochs=5, device="cuda")
report = calibrate("/content/head.val_logits.npz", "/content/head.calibration.json")
print(report)   # temperature, ECE before/after
# NOTE: this first run trains and validates within the GenImage val split —
# fine for pipeline validation; publishable numbers need train/ + val/ disjoint
# by the dataset's own construction.

train: 30000 samples (5.9 GB fp16), val: 2000 samples, dim=384, device=cuda
epoch 0: train_loss=0.2843 val_acc=0.9110
epoch 1: train_loss=0.1399 val_acc=0.9130
epoch 2: train_loss=0.0835 val_acc=0.9165
epoch 3: train_loss=0.0493 val_acc=0.9220
epoch 4: train_loss=0.0288 val_acc=0.9190
{'temperature': 2.5835, 'ece_before': 0.0521, 'ece_after': 0.0166, 'nll_before': 0.3116, 'nll_after': 0.2048}


## Step 4 — Persist artifacts to Drive, versioned by commit

Checkpoints are the artifact whose loss hurts; they're small. Version by commit
hash so every result is traceable to the exact code that produced it.

In [7]:
import json, shutil, subprocess
from pathlib import Path

commit = subprocess.run(["git", "-C", "/content/AI-image-Checkers", "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
RUN = Path(f"/content/drive/MyDrive/ai_image_id/runs/{commit}")
RUN.mkdir(parents=True, exist_ok=True)

for f in ["/content/head.pt", "/content/head.calibration.json",
          "/content/data/train/manifest.csv", "/content/data/val/manifest.csv"]:
    shutil.copy(f, RUN)

(RUN/"provenance.json").write_text(json.dumps({
    "commit": commit, "backbone": "dinov2_vits14",
    "data": "GenImage sdv14 val-split slice", "n_train_per_class": 5000,
    "n_val_per_class": 1000, "n_aug": 2, "seeds": {"prep": 0},
}, indent=2))
print("saved to", RUN)

saved to /content/drive/MyDrive/ai_image_id/runs/cb68637


## Step 5 — Use the detector in the pipeline

The fusion engine caps classifier evidence at `likely` (never `verified`),
down-weights on recompression drift, and a confidently-low score unlocks
`unlikely`.

In [10]:
import os, random
from ai_image_id.main import analyze_image

random.seed(0)
fake_dir, real_dir = "/content/data/val/fake", "/content/data/val/real"
samples = [os.path.join(fake_dir, random.choice(os.listdir(fake_dir))),
           os.path.join(real_dir, random.choice(os.listdir(real_dir)))]

for path in samples:
    r = analyze_image(path, detector_ckpt="/content/head.pt")
    d = r.evidence.detector
    print(f"\n{path.split('/')[-2]}/{path.split('/')[-1]}")
    print(f"  verdict: {r.ai_verdict.value} ({r.confidence})")
    print(f"  detector: p={d.p_calibrated}  drift={d.robustness_drift}")
    print(f"  notes: {r.notes}")


fake/fake_005172.jpg
  verdict: likely (0.9)
  detector: p=0.9307  drift=0.0051
  notes: ['learned detector p=0.93 (drift 0.01)']

real/real_005816.jpg
  verdict: unlikely (0.6)
  detector: p=0.0153  drift=0.0024
  notes: ['learned detector p=0.02 (low) — weak evidence of camera origin; PRNU/reverse-search needed to strengthen']


In [21]:
!git fetch origin
!git log --oneline -5
!git status
!git pull                  # fast-forwards (rebase config makes this clean)

d07714c (HEAD -> main, origin/main, origin/HEAD) Run thrown
a815bd6 Created using Colab
cb68637 Refactor training logic and memory handling
5d4d049 Created using Colab
833e6c4 Created using Colab
On branch main
Your branch is up to date with 'origin/main'.

Changes not staged for commit:
  (use "git add <file>..." to update what will be committed)
  (use "git restore <file>..." to discard changes in working directory)
	modified:   ai_image_id/__pycache__/__init__.cpython-312.pyc
	modified:   ai_image_id/__pycache__/forensics.cpython-312.pyc
	modified:   ai_image_id/__pycache__/fusion.cpython-312.pyc
	modified:   ai_image_id/__pycache__/ingest.cpython-312.pyc
	modified:   ai_image_id/__pycache__/main.cpython-312.pyc
	modified:   ai_image_id/__pycache__/provenance.cpython-312.pyc
	modified:   ai_image_id/__pycache__/schema.cpython-312.pyc
	modified:   ai_image_id/detector/__pycache__/__init__.cpython-312.pyc
	modified:   ai_image_id/watermark/__pycache__/__init__.cpython-312.pyc
	modifie

## Next sessions
1. **Cross-generator eval**: extract other generators' `val/*` slices (Midjourney,
   ADM, BigGAN, GLIDE — same 7z pattern trick), embed, and build the table with
   `training.calibrate_eval.cross_generator_table`. Baseline to beat: GenImage's
   ResNet-50 gets ~52–55% on unseen generators.
2. **Real train/ run**: materialize a random 20K/class from `train/` (HF mirror
   with per-file access, or one-time extraction on a big disk → re-upload the
   sample to Drive), train on it, validate on `val/` — disjoint by construction.
3. **ONNX export** + pin the detector into CI with a tiny fixture.

In [24]:
GENIMAGE = "/content/drive/MyDrive/genimage_shared"   # shortcut to the PARENT folder
!ls "$GENIMAGE"
# expect siblings like: ADM, BigGAN, glide, Midjourney, stable_diffusion_v_1_4, VQDM, wukong ...

ADM	glide	    stable_diffusion_v_1_4  VQDM
BigGAN	Midjourney  stable_diffusion_v_1_5  wukong


xtract → prepare → embed, one generator at a time, cleaning up as it goes (each extracted slice is ~3–4 GB; embeddings kept are ~400 MB per generator, so the loop stays comfortably inside disk):

In [30]:
import shutil, subprocess
from pathlib import Path
from training.prepare_data import prepare_split
from training.embed import precompute

GENS = ["Midjourney", "ADM", "BigGAN", "glide"]   # match the ls output exactly

for gen in GENS:
    src = Path(f"/content/drive/MyDrive/genimage_shared/{gen}")
    masters = sorted(src.glob("*.zip"))
    if not masters:
        raise FileNotFoundError(f"{gen}: no .zip master in {src} — check folder name / shortcut")
    master = masters[0]
    workdir = Path(f"/content/eval/{gen}")
    if (workdir/"emb").exists():
        print(f"{gen}: embeddings already present, skipping"); continue

    print(f"═══ {gen}: extracting val slice ═══")
    !7z x -y -o/content/eval/{gen}/raw "{master}" "*/val/*" | tail -3

    # locate the val dir regardless of the wrapper name inside the archive
    val = next(p for p in (workdir/"raw").rglob("val") if (p/"ai").is_dir())
    print("val dir:", val)

    prepare_split(val/"nature", val/"ai", workdir/"data", n_per_class=1_000, seed=42)
    precompute(workdir/"data/manifest.csv", workdir/"data", workdir/"emb",
               n_aug=0, device="cuda")

    shutil.rmtree(workdir/"raw"); shutil.rmtree(workdir/"data")   # keep only embeddings
    print(f"{gen}: done\n")

═══ Midjourney: extracting val slice ═══
Files: 12000
Size:       8328212761
Compressed: 227469216710
val dir: /content/eval/Midjourney/raw/imagenet_midjourney/val


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


wrote 63 shards to /content/eval/Midjourney/emb
Midjourney: done

═══ ADM: extracting val slice ═══
Files: 12000
Size:       1520044631
Compressed: 38839383692
val dir: /content/eval/ADM/raw/imagenet_ai_0508_adm/val


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


wrote 63 shards to /content/eval/ADM/emb
ADM: done

═══ BigGAN: extracting val slice ═══
Files: 12000
Size:       982096774
Compressed: 24235936500
val dir: /content/eval/BigGAN/raw/imagenet_ai_0419_biggan/val


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


wrote 63 shards to /content/eval/BigGAN/emb
BigGAN: done

═══ glide: extracting val slice ═══
Files: 12000
Size:       1256157232
Compressed: 32263862577
val dir: /content/eval/glide/raw/imagenet_glide/val


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main


wrote 63 shards to /content/eval/glide/emb
glide: done



In [31]:
import json, numpy as np, torch
from training.train_head import build_head, _load_split, _batched_logits
from training.calibrate_eval import cross_generator_table, _sigmoid

state = torch.load("/content/head.pt", map_location="cpu")
head = build_head(state["dim"]); head.load_state_dict(state["state_dict"])
head = head.to("cuda").eval()
T = json.load(open("/content/head.calibration.json"))["temperature"]

def score(emb_dir):
    p, c, y = _load_split(emb_dir)
    logits = _batched_logits(head, p, c, "cuda").numpy().astype(np.float64)
    return _sigmoid(logits / T), y.astype(np.float64)

results = {"sdv14 (in-dist)": score("/content/emb/val")}
for gen in GENS:
    results[gen] = score(f"/content/eval/{gen}/emb")

print(cross_generator_table(results))

| generator | AUROC | bal. acc |
|---|---|---|
| ADM | 0.335 | 0.460 |
| BigGAN | 0.580 | 0.522 |
| Midjourney | 0.842 | 0.706 |
| glide | 0.791 | 0.648 |
| sdv14 (in-dist) | 0.974 | 0.919 |
| **mean** | **0.704** | **0.651** |


In [ ]:
from google.colab import files
from ai_image_id.detector import analyze_detector
import numpy as np
from PIL import Image

up = files.upload()
for name in up:
    rgb = np.asarray(Image.open(name).convert("RGB"))
    d = analyze_detector(rgb, ckpt="/content/head.pt", device="cuda")
    print(f"{name}: p_calibrated={d.p_calibrated}  (human artwork — high p = false positive)")